# Register PRAGMA Pipeline on OpenShift AI

Compiles the PRAGMA pretraining pipeline to YAML and registers it with the
OpenShift AI pipeline server (Kubeflow Pipelines v2).

**Run this notebook once** to deploy the pipeline definition.  
Subsequent runs can be triggered from the RHOAI dashboard or via the KFP SDK.

### What this does
1. Adds the repo root to `sys.path` so the `pipeline` package is importable  
2. Compiles `pragma_pretraining_pipeline` → `pipeline/pragma_pipeline.yaml`  
3. Connects to the DSPA via the **cluster-internal service** (bypasses OAuth proxy)  
4. Uploads the compiled YAML as a new pipeline (or new version if it already exists)

### Prerequisites
- Workbench pod running (init container has cloned the repo)  
- DSPA pods running: `oc get pods -n pragma-encoder | grep ds-pipeline`

## Cell 1 — Imports and path setup

In [ ]:
import os
import sys
import pathlib

# The init container clones the repo to /opt/app-root/src/pragma-encoder.
# Add it to sys.path so `from pipeline.pragma_pipeline import ...` works.
REPO_ROOT = pathlib.Path("/opt/app-root/src/pragma-encoder")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"Repo exists: {REPO_ROOT.exists()}")

# Connect to the DSPA pipeline server via the cluster-internal service (port 8888).
# The internal endpoint bypasses the OAuth proxy — no bearer token needed.
# Port 8888 is HTTPS; the Notebook Controller injects the cluster CA bundle
# at /etc/pki/tls/custom-certs/ca-bundle.crt which covers the DSPA's cert.
#
# Do NOT use the external route (*.apps.*) — it requires an OpenShift OAuth
# session cookie, not a SA token, and returns a 403/login page to SDK clients.
KFP_HOST = "https://ds-pipeline-pipelines-definition.pragma-encoder.svc.cluster.local:8888"
KFP_CA_CERT = "/etc/pki/tls/custom-certs/ca-bundle.crt"

print(f"KFP host:    {KFP_HOST}")
print(f"CA cert:     {KFP_CA_CERT} (exists: {os.path.exists(KFP_CA_CERT)})")

## Cell 2 — Compile pipeline to YAML

In [ ]:
import kfp.compiler as compiler
from pipeline.pragma_pipeline import pragma_pretraining_pipeline

PIPELINE_YAML = str(REPO_ROOT / "pipeline" / "pragma_pipeline.yaml")

compiler.Compiler().compile(
    pipeline_func=pragma_pretraining_pipeline,
    package_path=PIPELINE_YAML,
)
print(f"Pipeline compiled → {PIPELINE_YAML}")

## Cell 3 — Connect to KFP server and upload pipeline

In [ ]:
import kfp

client = kfp.Client(
    host=KFP_HOST,
    verify_ssl=True,
    ssl_ca_cert=KFP_CA_CERT,
)
print("Connected to KFP server — OK")

In [ ]:
PIPELINE_NAME = "pragma-pretraining-pipeline"
PIPELINE_DESCRIPTION = (
    "End-to-end PRAGMA foundation model pretraining pipeline. "
    "Implements the masked event modelling (MEM) objective from "
    "Ostroukhov et al. (2026), arXiv:2604.08849v1."
)

# Check if pipeline already exists — upload a new version if so
existing = [p for p in client.list_pipelines().pipelines or [] if p.display_name == PIPELINE_NAME]

if existing:
    pipeline_id = existing[0].pipeline_id
    print(f"Pipeline '{PIPELINE_NAME}' already exists (id={pipeline_id}) — uploading new version")
    result = client.upload_pipeline_version(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_version_name="latest",
        pipeline_id=pipeline_id,
        description=PIPELINE_DESCRIPTION,
    )
else:
    print(f"Creating pipeline '{PIPELINE_NAME}'")
    result = client.upload_pipeline(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_name=PIPELINE_NAME,
        description=PIPELINE_DESCRIPTION,
    )

print(f"\nPipeline registered: {result.display_name}")
print(f"Pipeline ID:         {result.pipeline_id}")
print(f"\nOpen in RHOAI dashboard → Data Science Pipelines → {PIPELINE_NAME}")